# Dual-Source Reconciliation (Slice 5C)

SEC and FMP now feed the same `CanonicalFinancialHistory`. Before FMP could become the primary operational provider, we need evidence of where they agree and where they differ, why they differ, and whether the differences change OwnerLens economics.

```
SEC Company Facts ─→ sec_adapter ─┐
                                  ├─→ reconcile_histories ─→ fact rows + derived rows + verdict
FMP (local cache)  ─→ fmp_adapter ─┘
```

Ground rules:

* Only **canonical reported facts** are compared. OwnerLens recomputes FCF, FCF/share, margins, ROIC, and classifications from each history; FMP's own precomputed ratios are never used as truth.
* Tolerance is **rounding only**: a difference of at most $0.5M (or 0.5M shares) is `WITHIN_TOLERANCE`. Anything larger is `REVIEW` until a documented `KnownDiscrepancy` explains it, and an explained difference stays visible as `EXPLAINED`.
* SEC is retrieved live, which costs nothing, and its content hash is recorded. FMP is read **only** from the local raw-response cache, and this notebook refuses any FMP network call.

In [1]:
import json
from pathlib import Path

import httpx

from owner_lens import (
    FmpClient,
    FmpResponseCache,
    SecClient,
    canonical_history_from_fmp,
    canonical_history_from_sec,
    compute_content_hash,
    load_settings,
)
from owner_lens.known_discrepancies import KNOWN_DISCREPANCIES
from owner_lens.reconciliation import (
    ReconciliationStatus,
    format_reconciliation_report,
    reconcile_histories,
)

repo_root = next((p for p in (Path('.'), Path('..')) if (p / 'pyproject.toml').exists()), Path('.'))
env_file = repo_root / '.env'
settings = load_settings(env_file=str(env_file) if env_file.exists() else None)


def refuse_network(request: httpx.Request) -> httpx.Response:
    raise httpx.ConnectError('FMP network disabled: reconciliation is cache-only', request=request)


fmp = FmpClient(
    'cache-only',
    http_client=httpx.Client(transport=httpx.MockTransport(refuse_network)),
    cache=FmpResponseCache(repo_root / 'data' / 'raw' / 'fmp'),
)

TICKERS = ('ADBE', 'V', 'COST', 'MSFT')  # CRM and NOW are outside the free FMP plan
histories, reports = {}, {}
with SecClient(settings.sec_user_agent) as sec:
    for ticker in TICKERS:
        result = sec.retrieve_company_facts(ticker)
        content_hash = compute_content_hash(json.dumps(result.raw_facts, sort_keys=True).encode())
        sec_history = canonical_history_from_sec(result.raw_facts, ticker=ticker)
        fmp_history = canonical_history_from_fmp(fmp.retrieve_annual_statements(ticker))
        histories[ticker] = (sec_history, fmp_history)
        reports[ticker] = reconcile_histories(sec_history, fmp_history, explanations=KNOWN_DISCREPANCIES)
        print(f'{ticker:<5} SEC CIK {result.identity.cik}  content hash {content_hash[:16]}...')

ADBE  SEC CIK 0000796343  content hash 4241d1761736b282...


V     SEC CIK 0001403161  content hash 783a56fbf2223009...


COST  SEC CIK 0000909832  content hash 1c812fb0237b2852...


MSFT  SEC CIK 0000789019  content hash f8cdff440803fe34...


## 1. Summary across companies

Each canonical metric and fiscal year is one row. *Compared* counts rows where both providers have a value. `SEC_ONLY` and `FMP_ONLY` rows come mostly from window timing: SEC carries six balance-sheet years, while the free FMP tier returns five years per statement.

In [2]:
statuses = list(ReconciliationStatus)
print(f"{'':<6}{'compared':>9} " + ' '.join(f'{s.value[:10]:>10}' for s in statuses))
for ticker, report in reports.items():
    summary = report.summary()
    print(f'{ticker:<6}{summary.total_facts_compared:>9} ' + ' '.join(f'{summary.count(s):>10}' for s in statuses))

print()
print('Material differences by category:')
for ticker, report in reports.items():
    material = [r for r in report.facts if r.material]
    counts = {}
    for row in material:
        counts[row.category.value] = counts.get(row.category.value, 0) + 1
    print(f'  {ticker:<5} {counts or "none"}')

       compared      MATCH WITHIN_TOL     REVIEW  EXPLAINED   SEC_ONLY   FMP_ONLY SEMANTIC_D UNCOMPARAB
ADBE         80         76          3          0          2          6          0          0          0
V            75         73          0          0          2          5          5          5          0
COST         68         67          0          0          1         23         17          0          0
MSFT         80         71          0          0          9          5          5          0          0

Material differences by category:
  ADBE  {'PROVIDER_NORMALIZATION': 2}
  V     {'UNSUPPORTED': 5, 'PROVIDER_NORMALIZATION': 2, 'POLICY_DIFFERENCE': 5}
  COST  {'PROVIDER_NORMALIZATION': 1}
  MSFT  {'PROVIDER_NORMALIZATION': 4, 'UNSUPPORTED': 5, 'SEC_CONCEPT_SELECTION': 5}


## 2. Why do SEC and FMP disagree?

Every material difference carries provenance from both sides (the SEC concept, filing form, filed date, and accession; the FMP field and filed date) and a documented explanation with evidence. ADBE FY2024 is the case that prompted this slice: FMP-derived FCF is 7,824M against 7,873M from SEC.

In [3]:
print(format_reconciliation_report(reports['ADBE']))

ADBE — SEC vs FMP reconciliation
facts compared: 80  MATCH=76  WITHIN_TOLERANCE=3  REVIEW=0  EXPLAINED=2  SEC_ONLY=6  FMP_ONLY=0  SEMANTIC_DIFFERENCE=0  UNCOMPARABLE=0
verdict: NOT_ELIGIBLE

Metric                      FY          SEC          FMP        Diff  Status              Category               Reason
capital_expenditures      2024       183.0M       232.0M       49.0M  EXPLAINED           PROVIDER_NORMALIZATION difference +49,000,000 exceeds rounding tolerance; explained: FMP reports 232M of FY2024 capital expenditures; Adobe filed 183M. This 49M gap is the entire FCF difference (SEC 7,873M vs FMP 7,824M).
                               SEC: sec PaymentsToAcquirePropertyPlantAndEquipment end 2024-11-29 10-K filed 2025-01-13 0000796343-25-000004
                               FMP: fmp investmentsInPropertyPlantAndEquipment end 2024-11-29 filed 2025-01-13
                               evidence: SEC PaymentsToAcquirePropertyPlantAndEquipment FY2024 = 183M in both the 10-K filed 

The 49M FCF gap traces entirely to **capital expenditures**. Operating cash flow matches exactly (8,056M). Adobe's 10-Ks for FY2024 and FY2025 both report 183M of PP&E purchases for FY2024. FMP reports 232M and shows a matching +51M in `otherInvestingActivities`, so FMP reclassified investing cash flows away from the filed line items. The FMP value is explained but **not trustworthy** as an OwnerLens fact.

## 3. Visa: unsupported shares, policy differences, and two FMP deviations

In [4]:
v = reports['V']
for row in v.facts:
    if row.material and (row.fiscal_year in (2025, None)):
        sec_v = f'{row.sec_fact.value / 1e6:,.0f}M' if row.sec_fact else row.sec_status.value
        fmp_v = f'{row.fmp_fact.value / 1e6:,.0f}M' if row.fmp_fact else row.fmp_status.value
        print(f'{row.metric:<24} FY{row.fiscal_year}  SEC {sec_v:>20}  FMP {fmp_v:>10}  {row.status.value:<20} {row.category.value}')
        print(f'    {row.explanation.explanation if row.explanation else row.reason}')

diluted_shares           FY2025  SEC          UNSUPPORTED  FMP     1,966M  FMP_ONLY             UNSUPPORTED
    SEC Company Facts carry no undimensioned diluted-share count for Visa's multi-class structure, so the SEC path is UNSUPPORTED. FMP supplies an as-converted class A diluted count that cannot be verified against SEC.
repurchases              FY2025  SEC              18,316M  FMP    13,389M  EXPLAINED            PROVIDER_NORMALIZATION
    FMP reports 13,389M of FY2025 common-stock repurchases; Visa filed 18,316M. FMP appears to move about 4.9B into other financing activities.
cash                     FY2025  SEC              17,164M  FMP    20,154M  EXPLAINED            PROVIDER_NORMALIZATION
    FMP reports 20,154M of FY2025 cash and equivalents; Visa filed 17,164M. FMP's figure matches neither the filed cash line nor the restricted-inclusive total.
short_term_investments   FY2025  SEC  STRUCTURALLY_ABSENT  FMP     1,833M  SEMANTIC_DIFFERENCE  POLICY_DIFFERENCE
    OwnerLens ca

* **Diluted shares:** Visa's multi-class structure leaves SEC Company Facts with no undimensioned diluted-share count, so SEC is `UNSUPPORTED`. FMP supplies an as-converted count (1,966M for FY2025, equal to FMP net income divided by diluted EPS). FMP's diluted count equals its basic count in FY2022 and FY2023, so its share basis is inconsistent. The row stays `FMP_ONLY`; nothing is substituted into the SEC path.
* **Short-term investments:** OwnerLens policy deliberately treats Visa's investment securities as structurally absent. FMP reports them, and that is the documented reason Visa's FMP-backed net cash and ROIC differ from SEC.
* **FY2025 cash and repurchases:** FMP deviates from the filed 10-K. Cash is 20,154M against 17,164M filed, and repurchases are 13,389M against 18,316M filed. Earlier years match exactly.

## 4. COST: fiscal-year timing and lease reclassification

In [5]:
cost = reports['COST']
fy2026 = [r for r in cost.facts if r.fiscal_year == 2026]
print(f'FY2026 rows: {len(fy2026)}, statuses {sorted({r.status.value for r in fy2026})}')
print('  ', fy2026[0].explanation.evidence)
debt = next(r for r in cost.facts if r.metric == 'current_debt' and r.fiscal_year == 2025)
print()
print(f'current_debt FY2025: SEC {debt.sec_fact.value / 1e6:,.0f}M [{debt.sec_fact.provider_field}]  '
      f'FMP {debt.fmp_fact.value / 1e6:,.0f}M [{debt.fmp_fact.provider_field}]  -> {debt.status.value}')
print('  ', debt.explanation.evidence)

FY2026 rows: 17, statuses ['FMP_ONLY']
   FMP filingDate 2026-09-24 for period end 2026-08-30. The latest SEC Company Facts observation for COST is the 10-Q filed 2026-06-03.

current_debt FY2025: SEC 75M [LongTermDebtCurrent]  FMP 361M [shortTermDebt]  -> EXPLAINED
   SEC LongTermDebtCurrent FY2025 = 75M (10-K filed 2025-10-08, 0000909832-25-000101); OperatingLeaseLiabilityCurrent = 208M and FinanceLeaseLiabilityCurrent = 78M. FMP shortTermDebt = 361M = 75M + capitalLeaseObligationsCurrent 286M.


Costco's FY2026 ended 2026-08-30. FMP already has it, with a filing date of 2026-09-24, but Costco had not yet filed its FY2026 10-K with the SEC. These rows are `FMP_ONLY`/`PERIOD_TIMING`: they reflect timing, not a data error. They also show that FMP serves pre-10-K figures.

## 5. Do OwnerLens outputs agree?

The derived comparison recomputes OwnerLens outputs from each history and traces every difference back to the fact rows that cause it.

In [6]:
outputs = ('free_cash_flow', 'fcf_per_share', 'roic', 'annual_classification')
for ticker, report in reports.items():
    print(f'== {ticker}')
    for row in report.derived:
        if row.output in outputs + ('overall_classification',) and row.fiscal_year in (2025, 2024, None):
            fmt = lambda x: f'{x / 1e6:,.0f}M' if isinstance(x, int) else (f'{x:.4f}' if isinstance(x, float) else str(x))
            print(f'  {row.output:<23} FY{row.fiscal_year!s:<5} SEC {fmt(row.sec_value):>16}  FMP {fmt(row.fmp_value):>16}  {row.status.value}')

== ADBE
  free_cash_flow          FY2025  SEC           9,852M  FMP           9,852M  MATCH
  fcf_per_share           FY2025  SEC          23.0726  FMP          23.0726  MATCH
  free_cash_flow          FY2024  SEC           7,873M  FMP           7,824M  EXPLAINED
  fcf_per_share           FY2024  SEC          17.5072  FMP          17.3867  EXPLAINED
  roic                    FY2025  SEC           0.6157  FMP           0.6157  MATCH
  roic                    FY2024  SEC           0.4477  FMP           0.4477  MATCH
  annual_classification   FY2025  SEC        IMPROVING  FMP        IMPROVING  MATCH
  annual_classification   FY2024  SEC        IMPROVING  FMP        IMPROVING  MATCH
  overall_classification  FYNone  SEC        IMPROVING  FMP        IMPROVING  MATCH
== V
  free_cash_flow          FY2025  SEC          21,577M  FMP          21,577M  MATCH
  fcf_per_share           FY2025  SEC             None  FMP          10.9751  FMP_ONLY
  free_cash_flow          FY2024  SEC          18,69

* **ADBE:** FCF FY2024 differs (`EXPLAINED`, traced to capex). ROIC and the classifications match.
* **V:** ROIC differs in every year, traced to the short-term-investments policy and, for FY2025, FMP's cash. SEC cannot classify Visa's per-share economics because diluted shares are unsupported, while FMP can. The classifications therefore differ for a documented reason, not because FMP knows more.
* **COST:** FY2025 ROIC differs slightly (traced to FMP's lease-inclusive short-term debt). The overall classification differs (SEC `IMPROVING`, FMP `STRONGLY_IMPROVING`) because FMP's latest year is the pre-10-K FY2026; the difference traces to timing, not economics.
* **MSFT:** owner economics match exactly. SEC capital efficiency cannot be computed because Microsoft has no `DebtCurrent` concept, so ROIC and the classifications are *unassessed*, not assumed equal.

## 6. Can FMP become the primary provider?

The verdicts are defined explicitly:

* `ELIGIBLE`: no material differences.
* `ELIGIBLE_WITH_EXPLAINED_DIFFERENCES`: material differences exist, each is understood and documented, and none makes a core fact untrustworthy.
* `NOT_ELIGIBLE`: at least one material unresolved difference, a stale explanation, or a core fact from FMP that cannot be trusted.

In [7]:
for ticker, report in reports.items():
    verdict = report.verdict()
    print(f'{ticker:<5} {verdict.eligibility.value}')
    for item in verdict.blocking:
        print(f'      blocking:   {item}')
    for item in verdict.unassessed[:1]:
        print(f'      unassessed: {item[:110]}')
    print(f'      explained material rows: {len(verdict.explained_material)}, stale explanations: {len(report.unused_explanations)}')

ADBE  NOT_ELIGIBLE
      blocking:   capital_expenditures FY2024: core fact not trustworthy from FMP (PROVIDER_NORMALIZATION)
      explained material rows: 1, stale explanations: 0
V     NOT_ELIGIBLE
      blocking:   cash FY2025: core fact not trustworthy from FMP (PROVIDER_NORMALIZATION)
      explained material rows: 11, stale explanations: 0
COST  NOT_ELIGIBLE
      blocking:   current_debt FY2025: core fact not trustworthy from FMP (PROVIDER_NORMALIZATION)
      explained material rows: 0, stale explanations: 0
MSFT  ELIGIBLE_WITH_EXPLAINED_DIFFERENCES
      unassessed: capital_efficiency: SEC cannot compute: ConceptNotFoundError: No supported US-GAAP concept with a fiscal-year-
      explained material rows: 14, stale explanations: 0


**Conclusion:** FMP is **not yet eligible** to be the primary provider. In three of four companies, FMP deviates from a filed core fact (ADBE FY2024 capex, Visa FY2025 cash, Costco FY2025 current debt). These are provider normalizations that would silently change FCF, net cash, and ROIC. Microsoft is eligible with explained differences, and there FMP is *more* complete than the current SEC path: it supplies the noncurrent long-term debt and the current portion that SEC concept selection misses.

SEC remains the primary and audit provider. The SEC findings (Microsoft's `LongTermDebt` includes the current portion; `DebtCurrent` is absent) are recorded as follow-ups. They are not fixed in this slice.

In [8]:
# Reproducibility: nothing above called FMP. These are the cached raw responses used.
print(sorted(p.relative_to(fmp.cache.root).as_posix() for p in fmp.cache.root.rglob('*.json')))
fmp.close()

['balance-sheet-statement/ADBE.annual.limit5.json', 'balance-sheet-statement/COST.annual.limit5.json', 'balance-sheet-statement/MSFT.annual.limit5.json', 'balance-sheet-statement/V.annual.limit5.json', 'cash-flow-statement/ADBE.annual.limit5.json', 'cash-flow-statement/COST.annual.limit5.json', 'cash-flow-statement/MSFT.annual.limit5.json', 'cash-flow-statement/V.annual.limit5.json', 'income-statement/ADBE.annual.limit5.json', 'income-statement/COST.annual.limit5.json', 'income-statement/MSFT.annual.limit5.json', 'income-statement/V.annual.limit5.json']
